# Using the Eval Service for Standard Evaluation

Our setting is a standard evaluation where versions are fixed shards, and the agent is replaced by an
**orchestration plan**, each restricted to a subset of the gym's
tools that we generate beforehand and then execute ourselves.

|  | Tutorial (evolving) | This pipeline (standard) |
|---|---|---|
| Who runs the agent loop | **the service** (`react_agent` / `acp_codex_agent`) | **us** |
| Granularity | one agent, all tools | N subagents, each with its own tool allowlist |
| Planning | inside the harness |  an inspectable XML plan |
| Service's role | runs the agent **and** grades | **gym + grader only** |
| Evolving resource | central to the notebook | not used (fixed `(domain, version)`) |
| What we get back | the harness's result | full per-node trace + the same grade |

Grading is **identical** in both: `task.grade()` runs the task's hidden SQL verifiers against the
final database state. That is what makes the two comparable.

> Plan generation, XML parsing and dataset loading live elsewhere in the repo and are **not**
> shown here — the plan below is pasted in already-parsed, exactly as those steps would hand it
> over. Everything in this notebook is the eval service and nothing else.

## Setup

Only the SDK and `openai` are needed.

In [ ]:
# pip install:
#   WHEEL=$(curl -fsSL -H "Authorization: Bearer $EVAL_SERVICE_API_KEY" "$EVAL_SERVICE_URL/sdk" \
#           | python -c "import sys,json; print(json.load(sys.stdin)['path'])")
#   curl -fsSL -H "Authorization: Bearer $EVAL_SERVICE_API_KEY" "${EVAL_SERVICE_URL}${WHEEL}" -o "/tmp/${WHEEL##*/}"
#   pip install --upgrade --force-reinstall --no-deps "/tmp/${WHEEL##*/}"
#   pip install httpx openai tqdm
import json, os, re
from simple_agentic_evals import EvalClient, ServiceError, react_agent, to_openai_tools
from openai import OpenAI

# setdefault is correct for the URL: it must not clobber a value the shell exported.
os.environ.setdefault("EVAL_SERVICE_URL", "https://educator-marrow-cultural.ngrok-free.dev")
client = EvalClient(
    base_url=os.environ["EVAL_SERVICE_URL"],
    api_key=os.environ["EVAL_SERVICE_API_KEY"],   # -> Authorization: Bearer <token>
    timeout=1800,
)
oai    = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL  = "gpt-5"

# One EOG task, used by both modes below.
DATASET, BENCHMARK, DOMAIN = "evovling_agents", "eog", "itsm"
VERSION, SPLIT = 4, "train"
TASK_ID = "task_20260114_180601_485_99ba2325_2dc8b3ce"

In [ ]:
PLAN = [
  {"id": "find_l1_group", "agent": "user_group",
   "tools": ["list_user_groups", "add_new_user_group", "update_user_group"],
   "input": "Use list_user_groups to retrieve all groups. Filter to the single group that is "
            "IT Support AND classified as a level 1 support team. Return the group's unique ID "
            "and name."},

  {"id": "select_and_escalate_oldest_incident", "agent": "incident",
   "tools": ["list_incidents", "get_incidents_assigned_to", "update_incident",
             "find_incident_by_id", "find_incident_by_number"],
   "input": "Using ${find_l1_group}, call get_incidents_assigned_to for that group. Keep "
            "incidents whose status is not resolved/closed, sort by created_on ascending and "
            "take the oldest. Call update_incident to set impact=high, urgency=high, "
            "priority=critical. Return incident number, opened_by and assigned_to."},

  {"id": "fetch_users_for_notifications", "agent": "user",
   "tools": ["get_user", "get_user_using_email", "get_user_using_name", "list_users"],
   "input": "Using ${select_and_escalate_oldest_incident}, call get_user for the reporter and "
            "the assignee. Return each user's id, full name and email."},

  {"id": "send_escalation_notifications", "agent": "notification",
   "tools": ["send_notification"],
   "input": "Using ${fetch_users_for_notifications} and "
            "${select_and_escalate_oldest_incident}, send two notifications via "
            "send_notification: one to the reporter and one to the assignee, each greeting the "
            "person by full name and naming the escalated incident."},
]
# Topological order; the real parser derives this from the <edge> block.
ORDER = [n["id"] for n in PLAN]

### Open the task and connect to the gym

`client.task(...)` provisions a **fresh database**. `task.mcp_session(server)` is the acting
surface.

In [ ]:
task = client.task(DATASET, BENCHMARK, VERSION, TASK_ID, split=SPLIT, domain=DOMAIN)
task.__enter__()                       # kept open across the cells below

mcp        = task.mcp_session(task.mcp_servers[0])
all_tools  = mcp.list_tools()
by_name    = {t["name"]: t for t in all_tools}
print(f"{len(all_tools)} tools on {task.mcp_servers[0].name}")
print("task:", (task.user_prompt or ""), "...")

### Run the plan, node by node

Instead of handing the task to a harness, we walk the DAG:
each node gets its own short-lived ReAct loop, its own system prompt, and **only its own tools**.

Nodes run strictly sequentially, they all mutate one shared database, so concurrent writes
would make the grade non-reproducible.

In [ ]:
def run_node(node, upstream):
    """One subagent: its tools only, its instruction with ${refs} resolved."""
    instruction = re.sub(r"\$\{(\w+)\}",
                         lambda m: upstream.get(m.group(1), f"[{m.group(1)} unavailable]"),
                         node["input"])
    tools = to_openai_tools([by_name[t] for t in node["tools"] if t in by_name])
    messages = [
        {"role": "system", "content":
            f"You are the '{node['agent']}' specialist. Use ONLY your own tools. "
            f"Finish with a short report of what you did and the IDs you touched.\n\n"
            f"Overall task for context:\n{task.user_prompt or ''}"},
        {"role": "user", "content": instruction},
    ]
    for _ in range(12):                                  # per-node step cap
        msg = oai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools).choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            return msg.content or ""
        for tc in msg.tool_calls:                        # ← the only service-side acting
            out = mcp.call_tool(tc.function.name, json.loads(tc.function.arguments or "{}"))
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out)})
    return "[step cap reached]"


upstream = {}
for node_id in ORDER:
    node = next(n for n in PLAN if n["id"] == node_id)
    upstream[node_id] = run_node(node, upstream)
    print(f"[{node_id:36}] {len(node['tools'])} tools -> {upstream[node_id]}...")

Close MCP first, then grade. `task.grade()` ignores everything the agents *said* and runs the
task's hidden SQL verifiers against the **final database state**.

In [ ]:
mcp.close()
res = task.grade(keep_alive=True)

print(f"pass_rate={res.pass_rate:.2f}  ({res.n_passed}/{res.n_total})  success={res.overall_success}\n")
for v in res.per_verifier or []:
    print(f"  [{'PASS' if v['passed'] else 'FAIL'}] {v['name']:38} "
          f"expected={v['expected']} actual={v['actual']}")

task.__exit__(None, None, None)        # tear the session down

---
# ALE


### The plan

Same shape as the EOG plan above — produced upstream and parsed elsewhere. The specialists here
are ALE's installed agent types (`chemistry_molecular`, `devops_infra`, `runtime_base`) rather
than gym record-owners, and there is no per-node `tools` list: a specialist gets the sandbox, not
a tool allowlist.

In [ ]:
import json, os, re
from simple_agentic_evals import EvalClient, ServiceError, react_agent, to_openai_tools
from openai import OpenAI

os.environ.setdefault("EVAL_SERVICE_URL", "https://educator-marrow-cultural.ngrok-free.dev")
client = EvalClient(
    base_url=os.environ["EVAL_SERVICE_URL"],
    api_key=os.environ["EVAL_SERVICE_API_KEY"],   # -> Authorization: Bearer <token>
    timeout=1800,
)

In [ ]:
ALE_TASK_ID = "life_sciences/amber_minimization_script_prep_instance_1"
ALE_VERSION = 4

ALE_PLAN = [
  {"id": "amber_inputs", "agent": "chemistry_molecular",
   "desc": "Inspect the protein-complex PDB and draft Amber implicit-solvent inputs.",
   "input": "Inspect base/input/complex_structure.pdb. Determine practical Amber/tleap "
            "considerations visible from the staged PDB (residue naming compatibility, "
            "hydrogens/ions/waters/hetero atoms). Return the exact text content for leap.in and "
            "step2_implicit.mini.mdin using basename GLN_phb2_lc3_aurka_model_0. Do not write files."},

  {"id": "slurm_script", "agent": "devops_infra",
   "desc": "Draft the SLURM GPU submission script.",
   "input": "Prepare the exact text content for base/output/submit_min.sh but do not write any "
            "files. A SLURM submission script requesting one GPU with suitable CPU/memory/time "
            "for an Amber implicit-solvent minimization; load Amber 22 and CUDA 11.6.2 modules."},

  {"id": "final_writer_audit", "agent": "runtime_base",
   "desc": "Write the required output files exactly and audit the output directory.",
   "input": "Using the file contents returned by ${amber_inputs} and ${slurm_script}, create "
            "base/output if needed and write exactly these three files: leap.in, "
            "step2_implicit.mini.mdin, submit_min.sh. Create no other files. Preserve the "
            "basename GLN_phb2_lc3_aurka_model_0 consistently."},
]
ALE_ORDER = ["amber_inputs", "slurm_script", "final_writer_audit"]   # topological

In [ ]:
ALE_DATASET, ALE_BENCHMARK = "evovling_agents", "ale"

for ds in client.benchmarks().get("datasets", []):
    if ds.get("dataset") != ALE_DATASET:
        continue
    for bm in ds.get("benchmarks", []):
        if bm.get("benchmark") != ALE_BENCHMARK:
            continue
        for entry in bm.get("domains", []):
            name = entry.get("domain") or ALE_BENCHMARK
            print(f"domain={name!r}  versions={[v['version'] for v in entry.get('versions', [])]}")

ALE_DOMAIN = "ale"
ids = client.task_ids(ALE_DATASET, ALE_BENCHMARK, ALE_VERSION, split="train", domain=ALE_DOMAIN)
print(f"\n{len(ids)} task_ids in this slice; target present: {ALE_TASK_ID in ids}")

### Render the plan into the orchestrator prompt

`prompt_suffix` **replaces** the service's default orchestrator prompt rather than extending it,
and that default carries the software allowlist, the delegation protocol and the specialist
roster. So the task's own `system_prompt` is reproduced verbatim and the plan is appended to it.
Dropping it would leave a tool-less orchestrator unable to delegate at all.

In [ ]:
PLAN_HEADER = "## Orchestration plan (given — do not re-plan)"
PLAN_PREAMBLE = (
    "Protocol step (1) is already done for you. The delegation graph below was produced by an "
    "upstream planner and is the plan you must execute. Do not design your own decomposition "
    "and do not substitute specialists of your own choosing.\n\n"
    "Execute the nodes in the order listed. For each node, spawn the named specialist with that "
    "node's instruction, wait for it, and carry its result forward to the nodes that depend on "
    "it. Steps (2), (3) and (4) of the protocol still apply unchanged — in particular, confirm "
    "every required `output/` artifact was written by a specialist before you finish.\n\n"
    "If a node names a specialist that is not installed, say so explicitly in your final message "
    "and continue with the rest of the plan rather than silently re-planning around it."
)

def format_plan(plan, order):
    by_id = {n["id"]: n for n in plan}
    preds = {"final_writer_audit": ["amber_inputs", "slurm_script"]}   # from the plan's edges
    lines = [PLAN_HEADER, "", PLAN_PREAMBLE, ""]
    for i, nid in enumerate(order, 1):
        n = by_id[nid]
        lines.append(f"### Step {i} — `{nid}` → specialist `{n['agent']}`")
        lines.append(f"Role: {n['desc']}")
        if preds.get(nid):
            named = ", ".join(f"`{p}` ({by_id[p]['agent']})" for p in preds[nid])
            lines.append(f"Depends on: {named}. Pass their results into this step.")
        lines.append(f"Instruction: {n['input']}")
        lines.append("")
    lines.append(f"`{order[-1]}` is the final step. The task is not complete until every required "
                 "`output/` artifact exists on disk, written by a specialist.")
    return "\n".join(lines)


ale_task = client.task(ALE_DATASET, ALE_BENCHMARK, ALE_VERSION, ALE_TASK_ID,
                       split="train", domain=ALE_DOMAIN, resource_mode="accumulative")
ale_task.__enter__()
ale_task.start()                                   # provisions the Docker sandbox

system_prompt  = ale_task.system_prompt or ""      # allowlist + protocol + roster
prompt_suffix  = f"{system_prompt.rstrip()}\n\n{format_plan(ALE_PLAN, ALE_ORDER)}"
print(f"system_prompt {len(system_prompt)} chars + plan -> prompt_suffix {len(prompt_suffix)} chars")
print("\n" + format_plan(ALE_PLAN, ALE_ORDER))

In [ ]:
print(ALE_DATASET, ALE_BENCHMARK, ALE_DOMAIN, ALE_VERSION)

### Run Codex with our plan

`acp_codex_agent` is the tutorial's own ALE harness.
`prompt_suffix` replaces the orchestrator's default prompt, whose four-step protocol has
*"Plan which specialist handles each part of the task"* as step 1. Supplying our plan there, and
leaving every other part of that prompt byte-identical, turns a fresh Codex run into an
execution of the plan we already have.

In [ ]:
from simple_agentic_evals import acp_codex_agent
import hashlib

TIMEOUT_S = 7200
run = acp_codex_agent(
    ale_task,
    model="gpt-5.5",
    api_key=os.environ["OPENAI_API_KEY"],                     
    prompt_suffix=prompt_suffix,            
    max_episodes=4,
    timeout_s=TIMEOUT_S,
    require_completion=True,
    include_trace=True,
    verbose="steps",
)

print("completed:", run.completed, "| stopped:", run.stopped,
      "| episodes:", run.episodes, "| exit_code:", run.exit_code)
print("subagent spawns:", run.n_subagent_spawns, "| calls:", run.n_calls,
      "| tokens:", run.total_tokens)
print("final_message:", repr((run.final_message or "")), " <- usually empty on ALE")

if not run.total_tokens:
    print("\n!! 0 tokens — Codex never called the model. Run the next cell for the reason.")


### Artifacts, then grade

`task.fetch_run_artifacts()` returns the raw `ale_run` tree (trajectory, event log, and the
produced `output/`) as tar.gz. **Call it after the run but before grading**


In [ ]:
try:
    artifacts = ale_task.fetch_run_artifacts()      
    print(f"artifacts: {len(artifacts)} bytes")
except Exception as exc:
    artifacts = None
    print("artifacts unavailable:", type(exc).__name__)

res = ale_task.grade(keep_alive=True)             
print(f"\npass_rate={res.pass_rate}  success={res.overall_success}  ({res.n_passed}/{res.n_total})")
for v in res.per_verifier or []:
    print(f"  [{'PASS' if v['passed'] else 'FAIL'}] {v['name']}")
print("\n(ALE tasks are deliberately hard; a low score here is normal.)")

ale_task.__exit__(None, None, None)
